<a href="https://colab.research.google.com/github/mahieshwar-budati/Basic-Advance-RAG/blob/main/Primary_Key_Search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install "pymilvus[milvus_lite]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.3/55.3 MB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 301.2/301.2 kB 23.7 MB/s eta 0:00:00


Full Working Code- milvus

In [ ]:
from pymilvus import MilvusClient, DataType

# Connect (Milvus Lite)
client = MilvusClient(uri="./milvus_demo.db")

# Drop collection if exists
if client.has_collection("products"):
    client.drop_collection("products")

# Create schema
schema = client.create_schema(auto_id=True)

schema.add_field("id", DataType.INT64, is_primary=True)
schema.add_field("vector", DataType.FLOAT_VECTOR, dim=5)

# Prepare index
index_params = client.prepare_index_params()
index_params.add_index(
    field_name="vector",
    index_type="AUTOINDEX",
    metric_type="L2"
)

# Create collection
client.create_collection(
    collection_name="products",
    schema=schema,
    index_params=index_params
)

print("Collection created ✅")

# Insert sample data
data = [
    {"vector": [0.1, 0.2, 0.3, 0.4, 0.5]},
    {"vector": [0.2, 0.3, 0.4, 0.5, 0.6]},
    {"vector": [0.9, 0.8, 0.7, 0.6, 0.5]},
]

insert_result = client.insert(
    collection_name="products",
    data=data
)

print("Inserted IDs:", insert_result["ids"])

Collection created ✅
Inserted IDs: [464495661176061952, 464495661176061953, 464495661176061954]


Primary-Key Search
# Now we search using ID instead of vector.

In [ ]:
# Pick one existing ID
query_id = insert_result["ids"][0]

# Step 1: Get vector of that ID
entity = client.query(
    collection_name="products",
    filter=f"id == {query_id}",
    output_fields=["vector"]
)

query_vector = entity[0]["vector"]

# Step 2: Use that vector for similarity search
results = client.search(
    collection_name="products",
    data=[query_vector],
    limit=2
)

for hits in results:
    print("Similar products:")
    for hit in hits:
        print(hit)

Similar products:
{'id': 464495661176061952, 'distance': 0.0, 'entity': {}}
{'id': 464495748103012358, 'distance': 0.0, 'entity': {}}
